In [ ]:
import anndata as ad
import pandas as pd
import scipy.sparse as sp
import numpy as np
import os

sample_id = ["H20.33.002","H20.33.004","H20.33.012","H20.33.015","H20.33.025","H20.33.035","H20.33.040",
             "H20.33.044","H21.33.001","H21.33.005","H21.33.006","H21.33.011","H21.33.012","H21.33.013",
             "H21.33.014","H21.33.015","H21.33.016","H21.33.019","H21.33.021","H21.33.022","H21.33.023",
             "H21.33.025","H21.33.028","H21.33.031","H21.33.032","H21.33.038","H21.33.040"]

os.chdir(r"D:\Research_project\For Test\smoothie\smoothie_input")

for i in sample_id:
    counts_df = pd.read_csv(f"sample_{i}_counts.csv", index_col=0)
    spatial_df = pd.read_csv(f"sample_{i}_spatial.csv")

    adata = ad.AnnData(X=sp.csr_matrix(counts_df.values))
    adata.var_names = counts_df.columns
    adata.obs_names = counts_df.index
    adata.obsm["spatial"] = spatial_df.values  # numpy array, shape (n_cells, 2)

    adata.write_h5ad(f"{i}.h5ad")


f:\Anaconda\envs\scanpy_env\lib\site-packages\anndata\_core\anndata.py:812: UserWarning: 
AnnData expects .obs.index to contain strings, but got values like:
    [1201175622100280002, 1201175622100280004, 1201175622100280005, 1201175622100280006, 1201175622100280007]

    Inferred to be: integer

  names = self._prep_dim_index(names, "obs")
f:\Anaconda\envs\scanpy_env\lib\site-packages\anndata\_core\anndata.py:812: UserWarning: 
AnnData expects .obs.index to contain strings, but got values like:
    [1028044108100240734, 1028044108100240834, 1028044108100400012, 1028044108100400014, 1028044108100400019]

    Inferred to be: integer

  names = self._prep_dim_index(names, "obs")
f:\Anaconda\envs\scanpy_env\lib\site-packages\anndata\_core\anndata.py:812: UserWarning: 
AnnData expects .obs.index to contain strings, but got values like:
    [1206200522100500107, 1206200522100500108, 1206200522100500110, 1206200522100500112, 1206200522100500114]

    Inferred to be: integer

  names = self._

In [ ]:
import anndata as ad
import os

os.chdir(r"D:\Research_project\For Test\smoothie\smoothie_input")
adata = ad.read_h5ad("H20.33.004.h5ad")

print(adata)

print( adata.X.shape)

print( adata.obs_names[:5])

print(adata.var_names[:5])
print(adata.obsm["spatial"][:5])


In [ ]:
import anndata as ad
import numpy as np
import scanpy as sc
import sys
import os
import igraph

sys.path.append("D:\Research_project\For Test\smoothie\src")
from gaussian_smoothing import run_parallelized_smoothing
from spatial_correlation import compute_correlation_matrix
from network_analysis import make_spatial_network

os.chdir(r"D:/Research_project/For Test/smoothie/smoothie_input")

sample_id = ["H20.33.002","H20.33.004","H20.33.012","H20.33.015","H20.33.025","H20.33.035","H20.33.040",
             "H20.33.044","H21.33.001","H21.33.005","H21.33.006","H21.33.011","H21.33.012","H21.33.013",
             "H21.33.014","H21.33.015","H21.33.016","H21.33.019","H21.33.021","H21.33.022","H21.33.023",
             "H21.33.025","H21.33.028","H21.33.031","H21.33.032","H21.33.038","H21.33.040"]

for id in sample_id:
    
    print(f"processing {id}...")
    os.chdir(r"D:/Research_project/For Test/smoothie/smoothie_input")
    adata = ad.read_h5ad(f"{id}.h5ad")

    sc.pp.normalize_total(adata, target_sum=1e3)
    sc.pp.log1p(adata)

    sm_adata = run_parallelized_smoothing(
        adata,
        grid_based_or_not=False,      
        gaussian_sd=40.0,            
        min_spots_under_gaussian=40
    )

    pearsonR_mat, pval_mat = compute_correlation_matrix(sm_adata.X)


    edge_list, node_label_df = make_spatial_network(
        pearsonR_mat,
        gene_names=sm_adata.var_names,
        pcc_cutoff=0.4,          
        clustering_power=4,
        output_folder=r"D:\Research_project\For Test\smoothie\smoothie_output"
    )

    os.chdir(r"D:\Research_project\For Test\smoothie\smoothie_output")
    np.save(f"{id}_pearsonR.npy", pearsonR_mat)
    #edge_list.to_csv( f"{id}_edge_list.csv", index=False)
    #node_label_df.to_csv( f"{id}_node_labels.csv", index=False)
    print(f"{id} complete...")

print("all patients complete")


processing H20.33.002...


f:\Anaconda\envs\scanpy_env\lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Storage for smoothed count matrix will be 0.02 GB. 

Gaussian smoothing running on chunk 1 of 4... 
Gaussian smoothing running on chunk 2 of 4... 
Gaussian smoothing running on chunk 3 of 4... 
Gaussian smoothing running on chunk 4 of 4... 


f:\Anaconda\envs\scanpy_env\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


Total runtime for in-place Gaussian smoothing: 39.89 seconds.
compute_correlation_matrix running...
Total runtime for compute_correlation_matrix: 0.08677339553833008 seconds.
H20.33.002 complete...
processing H20.33.004...


...

Total runtime for in-place Gaussian smoothing: 53.58 seconds.
compute_correlation_matrix running...
Total runtime for compute_correlation_matrix: 0.062021732330322266 seconds.
H21.33.038 complete...
processing H21.33.040...


f:\Anaconda\envs\scanpy_env\lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Storage for smoothed count matrix will be 0.01 GB. 

Gaussian smoothing running on chunk 1 of 4... 
Gaussian smoothing running on chunk 2 of 4... 
Gaussian smoothing running on chunk 3 of 4... 
Gaussian smoothing running on chunk 4 of 4... 


f:\Anaconda\envs\scanpy_env\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


Total runtime for in-place Gaussian smoothing: 55.35 seconds.
compute_correlation_matrix running...
Total runtime for compute_correlation_matrix: 0.08130764961242676 seconds.
H21.33.040 complete...
all patients complete


In [ ]:
import numpy as np
import os
import pandas as pd

folder = r"D:\Research_project\For Test\smoothie\smoothie_output"
folder_1 = r"D:\Research_project\For Test\smoothie\smoothie_output\csv"
file_list = sorted([f for f in os.listdir(folder) if f.endswith("_pearsonR.npy")])

for fname in file_list:
    matrix = np.load(os.path.join(folder, fname))
    df = pd.DataFrame(matrix)

    csv_name = fname.replace(".npy", ".csv")
    df.to_csv(os.path.join(folder_1, csv_name), index=False)


In [21]:
!{sys.executable} -m pip install pyreadr

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 1.7 MB/s eta 0:00:01
   ------------------------------ --------- 1.0/1.4 MB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 2.0 MB/s eta 0:00:00
